# Data Description
- Author: RUEI-HSIANG, CHANG（AI協助：Codex）
- Date first created: 2026-11-14
- Background: 政黨光譜與黨員規模二維編碼。
- Unit of observation: 政黨、候選人、政黨有向配對，分別21、267、441列。
- Sample period: 候選人2026；黨員資料2023–2025及未標日期。
- Known issues: 不同口徑日期；100為使用者假設；新黨2000為近似值；無黨籍的100只是計算佔位；null光譜沿用舊距離特例。未重跑分群。
- Definition for each variable: 原始候選人欄位不變；party_members_used為計算人數；date/basis/status/source為來源與假設標記；party_spectrum_position為0–1光譜或空白；party_size_minmax=(人數−100)/331310；party_vector為JSON陣列[光譜,規模]；party_a/b為配對鍵，difference兩欄及JSON為兩維絕對差（光譜缺值用原特例）。
- Source data: `data/processed/candidates_scope_g1_g6_20261110.csv`
- Processing code: `data/code/prepare_party_vector_20261114.ipynb`
- Role: analysis input / lookup
- Verification status: clean kernel執行並重讀驗證，見以下輸出。
- Data path / SHA-256 / Downstream analysis input:
  - `data/processed/party_vector_20261114.csv`：`f57ecc5c3f01211ee526dc82c99a7eb6922bfaa9c307a1c5ca63ea29c27e795f`
  - `data/processed/candidates_scope_g1_g6_party_vector_20261114.csv`：`97b96b65b9e89a27db44768f1d4e3da91889c454b8efa5b0577231e358340f18`
  - `data/processed/party_difference_vector_20261114.csv`：`4d56330bc5628eac95054c12086baa388612e86421c4b6f4abf5e56bb15a22de`


## Change for Your Own Data
固定讀取下列三份資料並驗證hash。
## Summary
### Sample
### Summary Stats
### Assertions

In [1]:
import csv, json, hashlib
from pathlib import Path
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"AGENTS.md").exists())
FILES=[('party_vector_20261114.csv', 'f57ecc5c3f01211ee526dc82c99a7eb6922bfaa9c307a1c5ca63ea29c27e795f'), ('candidates_scope_g1_g6_party_vector_20261114.csv', '97b96b65b9e89a27db44768f1d4e3da91889c454b8efa5b0577231e358340f18'), ('party_difference_vector_20261114.csv', '4d56330bc5628eac95054c12086baa388612e86421c4b6f4abf5e56bb15a22de')]
for name, expected in FILES:
    p=ROOT/'data/processed'/name
    assert hashlib.sha256(p.read_bytes()).hexdigest()==expected
    with p.open(encoding='utf-8-sig',newline='') as f: rows=list(csv.DictReader(f))
    print(name, 'rows=',len(rows), 'columns=',list(rows[0]))
    print('Sample:',rows[:10])
    if 'difference' in name:
        assert len(rows)==441 and len({(r['party_a'],r['party_b']) for r in rows})==441
        pairs={(r['party_a'],r['party_b']):json.loads(r['party_difference_vector']) for r in rows}
        assert all(v==pairs[b,a] for (a,b),v in pairs.items())
        assert all(v==[0,0] for (a,b),v in pairs.items() if a==b)
    else:
        key='candidate_key' if name.startswith('candidates') else 'party_registration_raw'
        assert len({r[key] for r in rows})==len(rows)
        scores=[float(r['party_size_minmax']) for r in rows]
        assert min(scores)==0 and max(scores)==1
        assert all(abs(float(r['party_size_minmax'])-(int(r['party_members_used'])-100)/331310)<1e-12 for r in rows)
        assert all(len(json.loads(r['party_vector']))==2 for r in rows)
        print('Summary Stats:',len(rows),'min=',min(scores),'max=',max(scores),'imputed=',sum(r['party_members_status']=='imputed_100' for r in rows))
print('Assertions passed')


party_vector_20261114.csv rows= 21 columns= ['party_registration_raw', 'party_members_used', 'party_members_date', 'party_members_basis', 'party_members_status', 'party_members_source', 'party_spectrum_position', 'party_size_minmax', 'party_vector']
Sample: [{'party_registration_raw': '三勢團結促進聯盟', 'party_members_used': '100', 'party_members_date': '', 'party_members_basis': 'user_assumption', 'party_members_status': 'imputed_100', 'party_members_source': '', 'party_spectrum_position': '', 'party_size_minmax': '0.0', 'party_vector': '[null, 0.0]'}, {'party_registration_raw': '中國國民黨', 'party_members_used': '331410', 'party_members_date': '2025-09-10', 'party_members_basis': 'eligible_voters', 'party_members_status': 'reported', 'party_members_source': 'https://money.udn.com/money/story/7307/9029253', 'party_spectrum_position': '1.0', 'party_size_minmax': '1.0', 'party_vector': '[1.0, 1.0]'}, {'party_registration_raw': '勞動黨', 'party_members_used': '100', 'party_members_date': '', 'party_me